# 02 - API Integration with Ollama (Python)

This lab notebook focuses on **robust programmatic integration** with the Ollama REST API.

Topics:
- Health checks
- Listing installed models
- `/api/generate` and `/api/chat`
- Streaming responses (JSONL)
- Error handling and timeouts


In [ ]:
import json
import requests
from typing import Iterator, Dict, Any, List


In [ ]:
BASE_URL = 'http://localhost:11434'

def health_check() -> bool:
    try:
        r = requests.get(BASE_URL, timeout=3)
        print('Status:', r.status_code)
        return r.ok
    except requests.RequestException as e:
        print('Health check failed:', e)
        return False

health_check()


In [ ]:
def list_models() -> List[Dict[str, Any]]:
    r = requests.get(f'{BASE_URL}/api/tags', timeout=10)
    r.raise_for_status()
    return r.json().get('models', [])

models = list_models()
print('Installed models:', [m.get('name') for m in models])


In [ ]:
def generate(prompt: str, model: str = 'llama3.2:3b', temperature: float = 0.2) -> str:
    payload = {'model': model, 'prompt': prompt, 'temperature': temperature, 'stream': False}
    r = requests.post(f'{BASE_URL}/api/generate', json=payload, timeout=120)
    r.raise_for_status()
    return r.json().get('response', '')

print(generate('Explain overfitting in machine learning in 5 bullet points.'))


In [ ]:
def chat(messages: List[Dict[str, str]], model: str = 'llama3.2:3b', temperature: float = 0.2) -> str:
    payload = {'model': model, 'messages': messages, 'temperature': temperature, 'stream': False}
    r = requests.post(f'{BASE_URL}/api/chat', json=payload, timeout=120)
    r.raise_for_status()
    return r.json().get('message', {}).get('content', '')

msgs = [
    {'role': 'system', 'content': 'You are a concise teaching assistant.'},
    {'role': 'user', 'content': 'What is the bias-variance tradeoff?'}
]
print(chat(msgs))


In [ ]:
def generate_stream(prompt: str, model: str = 'llama3.2:3b') -> Iterator[str]:
    payload = {'model': model, 'prompt': prompt, 'stream': True}
    with requests.post(f'{BASE_URL}/api/generate', json=payload, stream=True, timeout=300) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            obj = json.loads(line.decode('utf-8'))
            yield obj.get('response', '')

text = ''
for chunk in generate_stream('Write a short explanation of transformers for engineers.'):
    text += chunk
print(text)
